# Multi-Agent Systems: Supervisor and Plan-and-Execute

One agent with a pile of tools and a long system prompt works, right up until it doesn't. This notebook builds the same underlying task two different ways, using a team of small, focused agents instead of one large one, and compares two ways of coordinating that team:

- **Supervisor** - a central router that delegates to specialists and collects their answers
- **Plan-and-Execute** - a planner that writes a checklist upfront, an executor that works through it one item at a time, and a replanner that decides when the job is actually done

We also build a small **Network / peer-to-peer** example at the end, the third pattern from the slides, since the deck names it but doesn't give it code.

Running scenario throughout: analyzing Q3 customer churn and producing a short written summary - a task that naturally splits into research, calculation, and writing.


## Setup

What: install the libraries we need.

Why: worth confirming these import cleanly on whatever version Colab hands you before relying on them.


In [1]:
!pip install -q langgraph langchain langchain-google-genai


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 5.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.


## API key

What: load the Gemini key into the environment.

Why: `ChatGoogleGenerativeAI` reads it from the environment rather than us passing it around as a plain string.


In [2]:
import os

try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")
except ImportError:
    import getpass
    if "GOOGLE_API_KEY" not in os.environ:
        os.environ["GOOGLE_API_KEY"] = getpass.getpass("Google API key: ")

print("key loaded")


key loaded


## Part 1: Why bother splitting into multiple agents

What: build the "everything in one prompt" version first, just to look at it, not to run it.

Why: "cognitive overload" is easy to say and easy to skip past. Seeing the actual size of a system prompt that tries to cover research, math, and writing all at once makes it concrete before we ever open LangGraph.


In [3]:
overloaded_system_prompt = '''
You are an assistant that can research topics, perform calculations, and write reports.
When asked to research something, search the web and summarize findings accurately.
When asked to calculate something, show your work and double check arithmetic.
When asked to write something, use a clear and professional tone, structure it into
paragraphs, and avoid jargon unless the user is clearly technical.
Do not mix these roles unless explicitly asked to. Always decide first which of the
three modes the request falls into before answering. If a request spans more than one
mode, handle them in the order: research, then calculation, then writing.
'''

overloaded_tools = [
    "search_web", "lookup_company_data", "read_pdf_report",
    "calculate_growth_rate", "calculate_churn_rate", "summarize_statistics", "run_python_snippet",
    "format_report", "generate_chart_description", "translate_text",
]

print(f"system prompt length: {len(overloaded_system_prompt)} characters")
print(f"tool count: {len(overloaded_tools)}")
print()
print("every single call to this agent sends the model all of the above,")
print("whether the request needs one tool or all ten of them.")


system prompt length: 647 characters
tool count: 10

every single call to this agent sends the model all of the above,
whether the request needs one tool or all ten of them.


Now compare that to three small agents, each with one job and two or three tools. Each one gets a short, focused prompt and a short tool list - less for the model to juggle, less room for it to reach for the wrong tool, and each one is independently testable. That split is what both patterns below are built on top of.


## Part 2: Three coordination patterns, in brief

- **Supervisor** - one LLM node decides who goes next, every worker reports back to it, it decides when the job is done. Centralized, predictable, easy to reason about and debug.
- **Plan-and-Execute** - a planner writes an explicit ordered checklist upfront, an executor works through it one step at a time, a replanner checks progress and can revise the plan. Good for longer, multi-stage goals where losing track of the overall plan is a real risk.
- **Network / peer-to-peer** - agents message each other directly, no central router. Decentralized, good for open-ended collaboration, harder to keep predictable.

We build the first two in full, then a small version of the third.


## Part 3: Define the specialist agents

What: three small `create_agent` instances - a researcher, an analyst, and a writer - each with its own short prompt and two focused tools.

Why mock tools again: same reasoning as the earlier notebooks, mock data keeps the session fast and independent of any external API's uptime or rate limits. Swap in real search/database calls for production.


In [4]:
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent


@tool
def search_web(query: str) -> str:
    """Search the web for information on a topic."""
    return (
        "Industry reports show average SaaS churn benchmarks sit between 5-7 percent "
        "monthly for mid-market companies, trending slightly down this year."
    )


@tool
def lookup_company_data(metric: str) -> str:
    """Look up an internal company metric by name."""
    data = {"customers_q2": "120", "customers_q3": "150", "churned_q3": "18"}
    return f"{metric}: {data.get(metric, 'not found')}"


@tool
def calculate_growth_rate(current: float, previous: float) -> str:
    """Calculate percentage growth between two values."""
    rate = round(((current - previous) / previous) * 100, 2)
    return f"growth rate: {rate}%"


@tool
def calculate_churn_rate(churned: float, total_start: float) -> str:
    """Calculate churn rate as a percentage."""
    rate = round((churned / total_start) * 100, 2)
    return f"churn rate: {rate}%"


model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

researcher_agent = create_agent(
    model,
    tools=[search_web, lookup_company_data],
    system_prompt="You are a research specialist. Find and report facts, do not calculate or write prose summaries.",
)

analyst_agent = create_agent(
    model,
    tools=[calculate_growth_rate, calculate_churn_rate],
    system_prompt="You are a data analyst. Perform calculations precisely and report the numbers, nothing else.",
)

writer_agent = create_agent(
    model,
    system_prompt="You are a writer. Turn the findings you are given into a short, clear, two-paragraph summary. Do not invent numbers that were not given to you.",
)


## Part 4: The supervisor's routing decision, under the hood

What: before wiring a graph, look at exactly what the supervisor produces - a structured decision, not free text.

Why: this mirrors the earlier function-calling notebook. `with_structured_output` forces the model's response to validate against our Pydantic schema, so `next_worker` is always one of our four literal options, never a typo or a sentence.


In [5]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder


class RouterDecision(BaseModel):
    next_worker: Literal["researcher", "analyst", "writer", "FINISH"] = Field(
        description="Who should act next, or FINISH if the overall task is complete."
    )
    instructions_for_worker: str = Field(
        description="A short, specific instruction for whichever worker is chosen next."
    )


supervisor_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You are the supervisor of a team: researcher, analyst, writer. "
        "Given the conversation so far, decide who should act next, and what exactly they "
        "should do. Pick FINISH only once the team has produced a full written summary."
    )),
    MessagesPlaceholder(variable_name="messages"),
])

supervisor_router = supervisor_prompt | model.with_structured_output(RouterDecision)

decision = supervisor_router.invoke({"messages": [
    ("user", "Research SaaS churn benchmarks, calculate our Q3 churn and growth from 120 to 150 customers, then write a two-paragraph summary.")
]})

print("next_worker:", decision.next_worker)
print("instructions:", decision.instructions_for_worker)


next_worker: researcher
instructions: Find SaaS churn benchmarks across different company sizes and industries.


## Part 5: Build the supervisor graph

What: a shared `TeamState`, a `supervisor` node that produces a `RouterDecision`, three worker nodes, and conditional edges that read `next_worker` to decide where to go.

Why the state needs a dedicated `next_worker` field: this is not part of the message history, it's routing information the graph itself needs to read between steps. Keeping it as its own field, separate from `messages`, is what makes the conditional edge below a one-line lookup instead of something that has to parse the last message.


In [6]:
%pip install grandalf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.4 MB/s eta 0:00:00


In [7]:
import operator
from typing import Annotated, TypedDict
from langchain_core.messages import AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages


class TeamState(TypedDict):
    # add_messages appends new messages and converts ("user", "...") tuples into message objects
    messages: Annotated[list, add_messages]
    next_worker: str
    instructions: str


def supervisor_node(state):
    decision = supervisor_router.invoke({"messages": state["messages"]})
    return {
        "next_worker": decision.next_worker,
        "instructions": decision.instructions_for_worker,
        "messages": [AIMessage(content=f"[supervisor] -> {decision.next_worker}: {decision.instructions_for_worker}")],
    }


def researcher_node(state):
    result = researcher_agent.invoke({"messages": [("user", state["instructions"])]})
    return {"messages": [AIMessage(content=f"[researcher] {result['messages'][-1].content}")]}


def analyst_node(state):
    result = analyst_agent.invoke({"messages": [("user", state["instructions"])]})
    return {"messages": [AIMessage(content=f"[analyst] {result['messages'][-1].content}")]}


def writer_node(state):
    result = writer_agent.invoke({"messages": [("user", state["instructions"])] + state["messages"]})
    return {"messages": [AIMessage(content=f"[writer] {result['messages'][-1].content}")]}


builder = StateGraph(TeamState)
builder.add_node("supervisor", supervisor_node)
builder.add_node("researcher", researcher_node)
builder.add_node("analyst", analyst_node)
builder.add_node("writer", writer_node)

builder.add_edge(START, "supervisor")
builder.add_conditional_edges("supervisor", lambda s: s["next_worker"], {
    "researcher": "researcher",
    "analyst": "analyst",
    "writer": "writer",
    "FINISH": END,
})
builder.add_edge("researcher", "supervisor")
builder.add_edge("analyst", "supervisor")
builder.add_edge("writer", "supervisor")

team_graph = builder.compile()

print(team_graph.get_graph().draw_ascii())


                                +-----------+                                 
                                | __start__ |                                 
                                +-----------+                                 
                                      *                                       
                                      *                                       
                                      *                                       
                               +------------+                                 
                               | supervisor |**                               
                         ******+------------+  ******                         
                   ******        ..        ...       *****                    
             ******            ..             .           ******              
          ***                 .                ..               ***           
+---------+          +------------+          +------

Notice the shape: one hub in the middle, every worker connects only to the supervisor, never to each other. Compare this to the simple two-node loop from the create_agent notebook, or the fan-out/fan-in shape from the parallel workflow demo - three genuinely different topologies for three genuinely different jobs.


## Part 6: Run the team, watch every handoff

What: invoke the graph on the full scenario and print every message in order, not just the final answer.

Why: the interesting part of a multi-agent run is the handoffs, not just the ending. Watching the supervisor route, a worker respond, and the supervisor route again is what makes the pattern click.


In [8]:
result = team_graph.invoke({
    "messages": [("user", (
        "Research SaaS churn benchmarks, calculate our Q3 churn and growth from 120 to "
        "150 customers with 18 churned, then write a two-paragraph summary."
    ))],
    "next_worker": "",
    "instructions": "",
})

for m in result["messages"]:
    print(m.content)
    print()


Research SaaS churn benchmarks, calculate our Q3 churn and growth from 120 to 150 customers with 18 churned, then write a two-paragraph summary.

[supervisor] -> researcher: Find current SaaS churn benchmarks across different company sizes and industries.

[researcher] [{'type': 'text', 'text': 'Average SaaS churn benchmarks sit between 5-7 percent monthly for mid-market companies, trending slightly down this year. I was unable to find more specific benchmarks across different company sizes and industries.', 'extras': {'signature': 'CqoHAWkUfRNrwXRjpaHC/s4rXb+UOCyXytLwNF5osKiZPQTlnfUBcZrJvucmOFX2aDb4QHfAKN4W5LdqIAYKRY4imL7FiQUzM8PwzPiSL1k3iWIpDHRA+MIYwwEHz6YTsQvmhGqEEhJihDy0jElTpaq4HJokfThiurVs3C6NKSXRBkHA0d3iII7jpm6KERpGRUFn8kCuJFuoBY1+TaOr9LJ8TIqnK0fnXjivRWGcdjrp+f8AlJcUHVwun8w9XEtOPX3OWErvsXq8bGUyPBUiEeTZr/mdaNzNnRPdbFZwOJFwcLOwOEpw+xf8+8V8Axg7xcDrPNggvSgo7qJOQmCbBSrgcyIWUET7ISvSuUJH7XdwcneQHYFBicDrfqgkI6QeDQ1fSMojcXi/La67ELOmrVqzb7puITEOsz/x7nZrrMJa1XvyQux4RLbvusq1XF6SZwb9DW9ubJBaV

## Part 6b: Single agent vs the team, measured

What: run the exact same task through one overloaded agent (all ten tools, the big system prompt from Part 1) and through the supervisor team, and measure both instead of arguing about them.

Why measure: Part 1 only printed a prompt length. That asserts a problem, it doesn't show one. Here we record every single model call each approach makes, including the calls happening inside the worker agents, and look at the numbers.

Two honest notes before running:
- On a small task like this, gemini-2.5-flash may well get the right answer as a single agent. Failures from overload are probabilistic and grow with task size, so don't expect a dramatic crash.
- What reliably differs is the shape of the work: how many calls, and how much context each call carries. That is what the table below shows.

If you are on a free-tier key and see a 429 / rate limit error, wait a minute and rerun the cell.


In [9]:
from langchain_core.callbacks import BaseCallbackHandler


# records the token usage of every model call made while it is attached
class CallLogger(BaseCallbackHandler):
    def __init__(self):
        self.calls = []

    def on_llm_end(self, response, **kwargs):
        message = response.generations[0][0].message
        usage = getattr(message, "usage_metadata", None) or {}
        self.calls.append(usage.get("input_tokens", 0))


# extra tools, deliberately similar to the existing ones, to recreate a crowded menu
@tool
def calculate_retention_rate(retained: float, total_start: float) -> str:
    """Calculate the percentage of customers retained over a period."""
    return f"retention rate: {round((retained / total_start) * 100, 2)}%"


@tool
def calculate_percentage_change(new_value: float, old_value: float) -> str:
    """Calculate the percentage change between an old value and a new value."""
    return f"percentage change: {round(((new_value - old_value) / old_value) * 100, 2)}%"


@tool
def summarize_statistics(numbers: list[float]) -> str:
    """Return the mean, min and max of a list of numbers."""
    return f"mean {sum(numbers) / len(numbers):.2f}, min {min(numbers)}, max {max(numbers)}"


@tool
def read_pdf_report(filename: str) -> str:
    """Read the text of an internal PDF report."""
    return "Q3 board report: customer base grew, churn flagged as a concern."


@tool
def format_report(text: str) -> str:
    """Format text into a clean report layout."""
    return text


@tool
def translate_text(text: str, language: str) -> str:
    """Translate text into another language."""
    return f"[{language}] {text}"


all_ten_tools = [
    search_web, lookup_company_data, read_pdf_report,
    calculate_growth_rate, calculate_churn_rate, calculate_retention_rate,
    calculate_percentage_change, summarize_statistics,
    format_report, translate_text,
]

# one agent holding every responsibility, using the big prompt from Part 1
single_agent = create_agent(model, tools=all_ten_tools, system_prompt=overloaded_system_prompt)

print(f"single agent: {len(all_ten_tools)} tools on every call")


single agent: 10 tools on every call


In [10]:
task = (
    "Research SaaS churn benchmarks, calculate our Q3 churn and growth from 120 to "
    "150 customers with 18 churned, then write a two-paragraph summary."
)

# run 1: the single overloaded agent
single_log = CallLogger()
single_result = single_agent.invoke(
    {"messages": [("user", task)]},
    config={"callbacks": [single_log]},
)

# run 2: the supervisor team from Part 5
team_log = CallLogger()
team_result = team_graph.invoke(
    {"messages": [("user", task)], "next_worker": "", "instructions": ""},
    config={"callbacks": [team_log]},
)


def describe(name, log):
    calls = log.calls
    print(f"{name}")
    print(f"  model calls:               {len(calls)}")
    print(f"  input tokens per call:     {calls}")
    print(f"  largest single call:       {max(calls)}")
    print(f"  average per call:          {sum(calls) // len(calls)}")
    print(f"  total input tokens:        {sum(calls)}")
    print()


describe("SINGLE AGENT", single_log)
describe("SUPERVISOR TEAM", team_log)


SINGLE AGENT
  model calls:               5
  input tokens per call:     [669, 728, 781, 836, 1095]
  largest single call:       1095
  average per call:          821
  total input tokens:        4109

SUPERVISOR TEAM
  model calls:               11
  input tokens per call:     [86, 116, 180, 249, 313, 852, 168, 270, 924, 976, 976]
  largest single call:       976
  average per call:          464
  total input tokens:        5110



Read the two blocks side by side. Things to look for:
- **input tokens per call** for the single agent tends to climb call after call, because every call resends the full history plus all ten tool schemas.
- the team's calls are individually smaller, since each worker only sees its own instructions and its own two tools.
- the team usually makes **more calls in total**, and its total token count can easily end up higher. That is not the demo failing, that is slide 8's cost trade-off showing up as a real number.

Now compare the actual final answers.


In [11]:
print("SINGLE AGENT FINAL ANSWER:")
print(single_result["messages"][-1].content)
print()

writer_messages = [m for m in team_result["messages"] if m.content.startswith("[writer]")]
print("TEAM FINAL ANSWER:")
print(writer_messages[-1].content if writer_messages else "(no writer output found)")


SINGLE AGENT FINAL ANSWER:
[{'type': 'text', 'text': 'Industry benchmarks for SaaS churn typically range from 5-7% monthly for mid-market companies. Our Q3 churn rate of 15.0% is significantly higher than this benchmark, indicating a need for immediate attention to customer retention strategies.\n\nDespite the high churn, our company experienced a robust customer growth of 25.0% in Q3, increasing from 120 to 150 customers. While this growth is positive, the elevated churn rate suggests that a substantial portion of new customer acquisition efforts are being offset by customer losses.', 'extras': {'signature': 'CqkBAWkUfRO42AL3JwwuOVcmI65TxwaMEby1YES6BgF3aigvi6ea0cONPqvhMu10L++2FGmMNHgOnj38IbFBtZghZcX5L3ZYyj0rYfSVCcl4lHxjPvH+zdP55Gk2fTeTk9QxGEGt0agdu8c6OSJ3C5+WQhFYvmU/xIvnD+AoxgEZftR32K1H9ZVXaaoe493sEvcEmr53EF/MQtrmsAosI4yBPX6AsmpQEd3hQQ=='}}]

TEAM FINAL ANSWER:
[writer]  Do not invent numbers that were not given to you.


### Tool confusion, tested directly

What: send the same five calculation questions to two setups, and check which tool each one reaches for.
- crowded menu: the model with all ten tools and the big prompt
- specialist: the model with only the analyst's two tools

Why these questions: every correct answer is either `calculate_churn_rate` or `calculate_growth_rate`, which both setups have. The crowded menu just also has near-lookalikes (`calculate_retention_rate`, `calculate_percentage_change`) sitting right next to them. Some wrong picks may be arguable, `calculate_percentage_change` is not a crazy choice for a growth question, and that ambiguity is exactly what a bigger menu creates.

We call the model once per question with `bind_tools`, same as the function calling notebook, no agent loop needed.


In [12]:
import time
from langchain_core.messages import SystemMessage, HumanMessage

# pause between calls to stay under free-tier rate limits, set to 0 on a paid key
PAUSE_SECONDS = 6

test_questions = [
    ("18 of our 120 customers left this quarter. What share did we lose?", "calculate_churn_rate"),
    ("Customer count went from 120 to 150. How much did we grow?", "calculate_growth_rate"),
    ("We started Q3 with 150 accounts and 12 cancelled. Work out the rate.", "calculate_churn_rate"),
    ("Accounts went from 80 last quarter to 96 this quarter, give me the percentage increase.", "calculate_growth_rate"),
    ("Out of 200 subscribers, 30 did not renew. Calculate the attrition.", "calculate_churn_rate"),
]

crowded_model = model.bind_tools(all_ten_tools)
specialist_model = model.bind_tools([calculate_growth_rate, calculate_churn_rate])


def picked_tool(response):
    return response.tool_calls[0]["name"] if response.tool_calls else "(no tool)"


crowded_score = 0
specialist_score = 0

for question, expected in test_questions:
    crowded = crowded_model.invoke([SystemMessage(overloaded_system_prompt), HumanMessage(question)])
    time.sleep(PAUSE_SECONDS)
    specialist = specialist_model.invoke([HumanMessage(question)])
    time.sleep(PAUSE_SECONDS)

    crowded_pick = picked_tool(crowded)
    specialist_pick = picked_tool(specialist)
    crowded_score += crowded_pick == expected
    specialist_score += specialist_pick == expected

    print(question)
    print(f"  expected:     {expected}")
    print(f"  crowded menu: {crowded_pick}")
    print(f"  specialist:   {specialist_pick}")
    print()

print(f"crowded menu matched expected: {crowded_score}/{len(test_questions)}")
print(f"specialist matched expected:   {specialist_score}/{len(test_questions)}")


18 of our 120 customers left this quarter. What share did we lose?
  expected:     calculate_churn_rate
  crowded menu: calculate_churn_rate
  specialist:   calculate_churn_rate

Customer count went from 120 to 150. How much did we grow?
  expected:     calculate_growth_rate
  crowded menu: calculate_growth_rate
  specialist:   calculate_growth_rate

We started Q3 with 150 accounts and 12 cancelled. Work out the rate.
  expected:     calculate_churn_rate
  crowded menu: calculate_churn_rate
  specialist:   calculate_churn_rate

Accounts went from 80 last quarter to 96 this quarter, give me the percentage increase.
  expected:     calculate_growth_rate
  crowded menu: calculate_growth_rate
  specialist:   calculate_growth_rate

Out of 200 subscribers, 30 did not renew. Calculate the attrition.
  expected:     calculate_churn_rate
  crowded menu: calculate_churn_rate
  specialist:   calculate_churn_rate

crowded menu matched expected: 5/5
specialist matched expected:   5/5


If both score 5/5, good - that is a real result too. It means at this size, gemini-2.5-flash handles a ten-tool menu fine, and splitting into a team would be premature (slide 8, point 3). Overload shows up as menus grow toward dozens of tools and prompts pile up conflicting rules, not at ten tools and five questions. The honest takeaway: measure before you split, don't split on principle.


## Part 7: Plan-and-Execute - the planner

What: a planner that produces an explicit, ordered checklist before any work happens, using structured output again so we get a real Python list back, not a paragraph we'd have to parse.

Why decouple planning from execution: a ReAct-style agent can lose track of the big picture once it's deep inside a tool call. Writing the whole plan down first means the agent always has something to check its progress against.


In [13]:
class Plan(BaseModel):
    steps: list[str] = Field(description="An ordered list of concrete steps needed to complete the goal.")


planner_prompt = ChatPromptTemplate.from_messages([
    ("system", "Break the user's goal into a short ordered list of concrete steps. Keep it to 3-5 steps."),
    ("user", "{goal}"),
])

planner = planner_prompt | model.with_structured_output(Plan)

plan = planner.invoke({"goal": "Analyze Q3 customer churn (120 to 150 customers, 18 churned) and produce a short report."})

for i, step in enumerate(plan.steps, 1):
    print(f"{i}. {step}")


1. Calculate the Q3 customer churn rate.
2. Summarize the key churn statistics for Q3.
3. Identify common potential factors contributing to customer churn.
4. Propose general recommendations to mitigate future churn.


## Part 8: Executor and replanner, wired into a loop

What: an executor that tackles exactly one step per turn using our specialist agents, and a replanner that looks at what's left and either sends control back to the executor or ends the run.

Why a separate replanner instead of just looping the executor directly: this is the seam where the plan could change. Right now it only checks "is the list empty", in a fuller version it would also decide whether earlier results mean the plan itself needs revising, not just whether to continue.


In [14]:
class ExecuteState(TypedDict):
    goal: str
    plan: list[str]
    completed: Annotated[list, operator.add]
    final_report: str


def plan_node(state):
    result = planner.invoke({"goal": state["goal"]})
    return {"plan": result.steps}


def execute_node(state):
    step = state["plan"][0]
    remaining_plan = state["plan"][1:]

    # route the step to whichever specialist fits, based on a simple keyword check
    if "research" in step.lower() or "benchmark" in step.lower():
        outcome = researcher_agent.invoke({"messages": [("user", step)]})
    elif "calculat" in step.lower() or "rate" in step.lower():
        outcome = analyst_agent.invoke({"messages": [("user", step)]})
    else:
        outcome = writer_agent.invoke({"messages": [("user", step)] + [
            ("assistant", c) for c in state["completed"]
        ]})

    outcome_text = outcome["messages"][-1].content
    return {"plan": remaining_plan, "completed": [f"[{step}] {outcome_text}"]}


def replan_node(state):
    if state["plan"]:
        return {}
    return {"final_report": state["completed"][-1]}


def route_after_replan(state):
    return "executor" if state["plan"] else END


plan_builder = StateGraph(ExecuteState)
plan_builder.add_node("planner", plan_node)
plan_builder.add_node("executor", execute_node)
plan_builder.add_node("replanner", replan_node)

plan_builder.add_edge(START, "planner")
plan_builder.add_edge("planner", "executor")
plan_builder.add_edge("executor", "replanner")
plan_builder.add_conditional_edges("replanner", route_after_replan, {"executor": "executor", END: END})

plan_execute_graph = plan_builder.compile()

print(plan_execute_graph.get_graph().draw_ascii())


+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
 +---------+   
 | planner |   
 +---------+   
      *        
      *        
      *        
+----------+   
| executor |   
+----------+   
      *        
      *        
      *        
+-----------+  
| replanner |  
+-----------+  
      .        
      .        
      .        
 +---------+   
 | __end__ |   
 +---------+   


The diagram draws as a straight line because the ASCII layout doesn't show the loop-back edge clearly, but it is there - watch the trace below, `executor` and `replanner` will each run once per step in the plan, not once total.


In [15]:
result = plan_execute_graph.invoke({
    "goal": "Analyze Q3 customer churn (120 to 150 customers, 18 churned) and produce a short report.",
    "plan": [],
    "completed": [],
    "final_report": "",
})

for step_result in result["completed"]:
    print(step_result)
    print()

print("FINAL REPORT:")
print(result["final_report"])


[Calculate the Q3 customer churn rate.] [{'type': 'text', 'text': 'I need the number of customers churned in Q3 and the total number of customers at the start of Q3 to calculate the churn rate.', 'extras': {'signature': 'CtwHAWkUfRNsIkDrrFBadCRdKLSiy8DQGg9WOGeUFoKbDfspJpo+Wa+ILLYO70AxoBXJB3TAqCL3da1FQKhDy+/j5DMUNoF+oM4S8JxYJ74l/JJ5vsyugkmEgtK/I1qqWJRqyCJvsgAqynwwOR7LIMvncDTuJz5Z6IT81+m/8ID5jsgmKZpO9La/SRpvj3vOES0A0Yt5gNAF9S4JGFHN2RJ2yj8GWD3LtD1oYgisuNhQzqoq90PKqbQ7RGafea00e8FgXcj/IyO29dCL4ZvT90wCeK2Nz0tNw7P68haEJr7zmoX8RL5xqFi/loZkkT51nWZHyvRJhthZcif5QeHwMio4MYBkWVkSOOa/F7Qn11kZZLn4amCT/v474+t1CzqGsrFsnioNfjp0u9PZ78GNjBJOWoxtmd8mlBvFUFQfLRfJmHyP8kDZVOkJW1GfAHDCIVdhyRzMfJh/4dpbiWzRHeBLrVVxEHCS0yJbBf8cLEUn7z2kV0XlLAflU031UVEodi+4QNCEkvP37227fvO7wAP7ScHv/z7eN2UvfwYg5kJI5GEjTPhZcp6Wb35BIhHK46sVJTpM1AuZAYjuMVO5aaBuHvU6aj9QUH42jj7Nv2cStVOEHwM4YRuaMeMA4dl34iUeul802os+n2lDxIefUjJ1WrpHGArO6TIL4NNq6p3zC23sQKfoTcz/F48rug5d2lbohy6EsKKW3oesi4Wob7YR7goVRt4HPZXsaX7bp7PPab/ysh0jlCGdKxgJYtZ1tbL1qmnd6VW

## Part 9: Network / peer-to-peer, briefly

What: the third pattern from the slides, no central router at all - two agents pass messages directly to each other for a fixed number of turns.

Why show this even briefly: it's the natural contrast to both patterns above. No hub means no single point of control or failure, but also nothing keeping the conversation on track - here we just cap the number of turns, a real system would need a smarter stopping condition.


In [ ]:
class NetworkState(TypedDict):
    messages: Annotated[list, add_messages]
    turns_left: int


def agent_a_node(state):
    result = researcher_agent.invoke({"messages": state["messages"]})
    reply = result["messages"][-1].content
    return {"messages": [AIMessage(content=f"[agent A - researcher] {reply}")], "turns_left": state["turns_left"] - 1}


def agent_b_node(state):
    result = analyst_agent.invoke({"messages": state["messages"]})
    reply = result["messages"][-1].content
    return {"messages": [AIMessage(content=f"[agent B - analyst] {reply}")], "turns_left": state["turns_left"] - 1}


def route_network(state):
    if state["turns_left"] <= 0:
        return END
    # alternate based on how many turns have been taken so far
    return "agent_a" if state["turns_left"] % 2 == 0 else "agent_b"


network_builder = StateGraph(NetworkState)
network_builder.add_node("agent_a", agent_a_node)
network_builder.add_node("agent_b", agent_b_node)
network_builder.add_conditional_edges(START, route_network, {"agent_a": "agent_a", "agent_b": "agent_b", END: END})
network_builder.add_conditional_edges("agent_a", route_network, {"agent_a": "agent_a", "agent_b": "agent_b", END: END})
network_builder.add_conditional_edges("agent_b", route_network, {"agent_a": "agent_a", "agent_b": "agent_b", END: END})

network_graph = network_builder.compile()

result = network_graph.invoke({
    "messages": [("user", "Sanity check our Q3 churn number of 12 percent against industry benchmarks.")],
    "turns_left": 4,
})

for m in result["messages"]:
    print(m.content)
    print()


## Part 10: Making the cost trade-off concrete

What: count how many separate LLM calls the supervisor run actually made.

Why: slide 8's "token costs & latency" point is easy to nod along to and forget. A number makes it stick - every handoff is a full model turn, and this adds up fast on longer tasks.


In [ ]:
# re-run the supervisor team with the CallLogger from Part 6b attached,
# so we count every model call, including the ones inside each worker agent
cost_log = CallLogger()
counting_result = team_graph.invoke(
    {"messages": [("user", task)], "next_worker": "", "instructions": ""},
    config={"callbacks": [cost_log]},
)

supervisor_turns = sum(1 for m in counting_result["messages"] if m.content.startswith("[supervisor]"))
worker_turns = sum(
    1 for m in counting_result["messages"]
    if m.content.startswith(("[researcher]", "[analyst]", "[writer]"))
)

print(f"supervisor decisions made: {supervisor_turns}")
print(f"worker handoffs made:      {worker_turns}")
print(f"total model calls, including calls inside workers: {len(cost_log.calls)}")
print(f"total input tokens: {sum(cost_log.calls)}")
print()
print("each worker handoff costs more than one model call, because every worker")
print("runs its own small agent loop inside. that multiplication is the real price")
print("of specialization - worth it past a certain complexity, wasteful before it.")


## Recap

**Supervisor** - centralized, predictable, easy to debug and extend with a new worker. Best when you can name your specialists upfront and the task naturally delegates.

**Plan-and-Execute** - keeps a long-horizon goal from getting lost mid-execution, and gives you an explicit place (the replanner) to adapt when things don't go as expected. Best for multi-stage goals where losing the thread is a real risk.

**Network / peer-to-peer** - no single point of control, most flexible, least predictable. Best for open-ended collaboration where you genuinely don't know the right structure upfront, worst when you need reliability or a clean audit trail.

None of these are free. Every pattern here trades latency and token cost for something - specialization, plan-adherence, or flexibility. Slide 8's third point is the one to leave the room with: don't reach for multi-agent on a simple, single-turn task. Start with one well-prompted agent, and split only once tool count or role conflict actually forces the issue.
